In [ ]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: vpn issue

🧠 Agent received input: vpn issue
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: vpn issue
        Category: network
        Priority: medium
        
Enter your query: exit
👋 Exiting MCP demo...


In [ ]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata

# Load API key securely

from google.colab import userdata

api_key = userdata.get('API_KEY')


client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
 Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM MCP Helpdesk Started (type 'exit')

Enter issue: Intenet Issue

🧠 Agent received: Intenet Issue
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'Intenet Issue', 'priority': 'medium', 'category': 'general'}

 Ticket Created Successfully!

Ticket ID: INC1000
Issue: Intenet Issue
Category: general
Priority: medium

Enter issue: exit
👋 Exiting...


In [ ]:
models = client.models.list()

for m in models.data:
    print(m.id)

openai/gpt-oss-20b
qwen/qwen3-32b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
groq/compound-mini
llama-3.3-70b-versatile
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
groq/compound
llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
moonshotai/kimi-k2-instruct
allam-2-7b
whisper-large-v3


MCP SCENARIO: “Smart HR Onboarding Assistant”
🧩 Scenario Background
You are working in a company called XYZ Corp.
New employees often face challenges during onboarding, such as:
- Trouble accessing payroll portal
- Confusion about leave policies
- Difficulty setting up email accounts
- Questions about training schedules
👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

🤖 What this Bot Should Do
When a new hire types a question/problem:
- Understand the query (e.g., “I can’t log into payroll”)
- Decide if escalation to HR is needed
- Identify:
- Category (Payroll / Policy / IT Setup / Training)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, step-by-step instructions)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [ ]:
import json
from groq import Groq
from google.colab import userdata

# Initialize Client
api_key = userdata.get('API_KEY')
client = Groq(api_key=api_key)

# ============================================
# STEP 0: HR KNOWLEDGE BASE & DB
# ============================================
hr_handbook = {
    "payroll": "To access the payroll portal, go to 'hr.corp.com' and use your SSO login. Initial setup takes 24 hours.",
    "leave": "New hires get 15 days of PTO. Requests must be submitted via the portal 2 weeks in advance.",
    "email": "Email setup is automatic. If it's been 2 hours and you can't log in, contact IT at ext 555."
}

hr_tickets = []

# ============================================
# STEP 1: TOOLS (The "Actions")
# ============================================

def get_instant_guidance(topic):
    """Tool to provide immediate answers from the handbook."""
    return hr_handbook.get(topic.lower(), "I couldn't find a specific guide, but I can open an HR ticket for you.")

def create_hr_ticket(issue, category, priority):
    """Tool to escalate to a human HR representative."""
    ticket_id = f"HR-{2000 + len(hr_tickets)}"
    ticket = {"id": ticket_id, "issue": issue, "category": category, "priority": priority}
    hr_tickets.append(ticket)
    return ticket

# ============================================
# STEP 2: LLM REASONING
# ============================================

def analyze_onboarding_query(user_input):
    prompt = f"""
    You are an HR Onboarding Assistant. Analyze the user query.

    1. Decide if you can answer using the handbook (Payroll, Leave, Email).
    2. Decide if a human HR ticket is needed (Escalation).

    Respond in JSON:
    {{
      "can_answer_instantly": true/false,
      "topic": "payroll/leave/email/unknown",
      "needs_ticket": true/false,
      "category": "Payroll/Policy/IT Setup/Training",
      "priority": "high/medium"
    }}

    User Query: "{user_input}"
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# ============================================
# STEP 3: MCP ORCHESTRATOR
# ============================================

def hr_agent(user_input):
    print(f"\n👋 Hello! Processing query: '{user_input}'")

    analysis = analyze_onboarding_query(user_input)

    # Logic Branch 1: Instant Guidance
    if analysis["can_answer_instantly"] and analysis["topic"] != "unknown":
        guide = get_instant_guidance(analysis["topic"])
        return f"💡 **Instant Guidance:** {guide}"

    # Logic Branch 2: Escalation
    if analysis["needs_ticket"]:
        ticket = create_hr_ticket(user_input, analysis["category"], analysis["priority"])
        return f"""
        ⚠️ **Issue Escalated to HR**
        I've created a ticket so a specialist can help you.
        Ticket ID: {ticket['id']}
        Priority: {ticket['priority']}
        Next Step: An HR rep will contact you within 4 hours.
        """

    return "🤖 I'm sorry, I'm not sure how to help with that. Should I connect you to a live chat?"

# ============================================
# RUN DEMO
# ============================================
print("🚀 HR Onboarding Bot Active (Type 'exit' to stop)\n")

while True:
    query = input("How can I help with your onboarding? ")
    if query.lower() == "exit": break

    print(hr_agent(query))

🚀 HR Onboarding Bot Active (Type 'exit' to stop)

How can I help with your onboarding? VPN Issue

👋 Hello! Processing query: 'VPN Issue'

        ⚠️ **Issue Escalated to HR**
        I've created a ticket so a specialist can help you.
        Ticket ID: HR-2000
        Priority: high
        Next Step: An HR rep will contact you within 4 hours.
        
How can I help with your onboarding? Today

👋 Hello! Processing query: 'Today'

        ⚠️ **Issue Escalated to HR**
        I've created a ticket so a specialist can help you.
        Ticket ID: HR-2001
        Priority: medium
        Next Step: An HR rep will contact you within 4 hours.
        
How can I help with your onboarding? Technical Access Issues

👋 Hello! Processing query: 'Technical Access Issues'

        ⚠️ **Issue Escalated to HR**
        I've created a ticket so a specialist can help you.
        Ticket ID: HR-2002
        Priority: high
        Next Step: An HR rep will contact you within 4 hours.
        
How can I 

MCP SCENARIO: “Smart Banking Support Assistant”
🧩 Scenario Background
You are working in a company called FinTrust Bank.
Customers often face issues such as:
- Credit card not working
- Trouble with online banking login
- Queries about loan status
- Transaction disputes
👉 Instead of calling customer care, customers use an AI Banking Support Bot.

🤖 What this Bot Should Do
When a customer types a problem:
- Understand the issue (e.g., “My card was declined”)
- Decide if escalation to a human agent is needed
- Identify:
- Category (Card Services / Online Banking / Loans / Transactions)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, troubleshooting steps, policy info)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [ ]:
import uuid
from datetime import datetime

# ------------------------------
# Step 1: Intent + Category Detection
# ------------------------------
def detect_category_and_intent(user_input):
    text = user_input.lower()

    if "card" in text or "declined" in text:
        return "Card Services", "Card Issue"
    elif "login" in text or "password" in text:
        return "Online Banking", "Login Issue"
    elif "loan" in text:
        return "Loans", "Loan Query"
    elif "transaction" in text or "fraud" in text:
        return "Transactions", "Transaction Issue"
    else:
        return "General", "Unknown"


# ------------------------------
# Step 2: Priority Assignment
# ------------------------------
def assign_priority(category, user_input):
    text = user_input.lower()

    if "fraud" in text or "unauthorized" in text:
        return "High"
    elif category in ["Card Services", "Transactions"]:
        return "High"
    else:
        return "Medium"


# ------------------------------
# Step 3: Escalation Decision
# ------------------------------
def needs_escalation(priority):
    return priority == "High"


# ------------------------------
# Step 4: Ticket Creation
# ------------------------------
def create_ticket(user_input, category, priority):
    ticket_id = str(uuid.uuid4())[:8]

    ticket = {
        "ticket_id": ticket_id,
        "issue": user_input,
        "category": category,
        "priority": priority,
        "status": "Open",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return ticket


# ------------------------------
# Step 5: Instant Response Generator
# ------------------------------
def generate_response(category):
    responses = {
        "Card Services": "Please check if your card is active and has sufficient balance. You can also try using it at another terminal.",
        "Online Banking": "Try resetting your password using 'Forgot Password'. Ensure caps lock is off.",
        "Loans": "You can check your loan status in the 'My Loans' section of the app.",
        "Transactions": "We recommend checking recent transactions. If unauthorized, block your card immediately.",
        "General": "Please provide more details so we can assist you better."
    }

    return responses.get(category, "We are processing your request.")


# ------------------------------
# Step 6: Main Bot Function (MCP Flow)
# ------------------------------
def banking_support_bot(user_input):
    print("\nUser Query:", user_input)

    # M → Model understanding
    category, intent = detect_category_and_intent(user_input)

    # P → Process logic
    priority = assign_priority(category, user_input)
    escalation = needs_escalation(priority)

    # C → Context used here (user input + classification)
    response = generate_response(category)

    print("\n--- AI RESPONSE ---")
    print(f"Category: {category}")
    print(f"Intent: {intent}")
    print(f"Priority: {priority}")
    print(f"Escalation Needed: {escalation}")
    print("\nGuidance:", response)

    if escalation:
        ticket = create_ticket(user_input, category, priority)
        print("\n🎫 Support Ticket Created:")
        for key, value in ticket.items():
            print(f"{key}: {value}")
    else:
        print("\n✅ Issue handled without escalation.")

    print("\n--------------------------\n")


# ------------------------------
# Run Example
# ------------------------------
if __name__ == "__main__":
    while True:
        user_query = input("Enter your issue (or 'exit'): ")
        if user_query.lower() == "exit":
            break
        banking_support_bot(user_query)

Enter your issue (or 'exit'): Card Services Issues

User Query: Card Services Issues

--- AI RESPONSE ---
Category: Card Services
Intent: Card Issue
Priority: High
Escalation Needed: True

Guidance: Please check if your card is active and has sufficient balance. You can also try using it at another terminal.

🎫 Support Ticket Created:
ticket_id: 9c603cfb
issue: Card Services Issues
category: Card Services
priority: High
status: Open
created_at: 2026-03-28 06:55:05

--------------------------

Enter your issue (or 'exit'): Loan Queries

User Query: Loan Queries

--- AI RESPONSE ---
Category: Loans
Intent: Loan Query
Priority: Medium
Escalation Needed: False

Guidance: You can check your loan status in the 'My Loans' section of the app.

✅ Issue handled without escalation.

--------------------------

Enter your issue (or 'exit'): exit


Create a  Weather Tool MCP Server that any AI agent can use with sample use case

In [ ]:
# ==============================
# MCP WEATHER TOOL SERVER + AGENT (ALL-IN-ONE)
# ==============================

from fastapi import FastAPI
import requests
import threading
import uvicorn
import time

# ------------------------------
# CONFIG (IMPORTANT)
# ------------------------------
from google.colab import userdata
api_key = userdata.get('weather_key')
API_KEY = api_key

# ------------------------------
# MCP TOOL SERVER
# ------------------------------
app = FastAPI()

@app.get("/get_weather")
def get_weather(city: str):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)

    if response.status_code != 200:
        return {"status": "error", "message": "City not found"}

    data = response.json()

    return {
        "status": "success",
        "data": {
            "city": city,
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"],
            "condition": data["weather"][0]["description"]
        }
    }

# ------------------------------
# RUN SERVER IN BACKGROUND
# ------------------------------
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)  # give server time to start

# ------------------------------
# AI AGENT (MCP FLOW)
# ------------------------------
def weather_agent(user_query):
    user_query = user_query.lower()

    # M → Model (intent detection)
    if "weather" in user_query:
        words = user_query.split()
        city = words[-1]

        # P → Process (call MCP tool)
        try:
            response = requests.get(
                f"http://127.0.0.1:8000/get_weather?city={city}"
            ).json()
        except:
            return "⚠️ Server not running."

        # C → Context (use tool output)
        if response["status"] == "success":
            data = response["data"]
            return (
                f"🌤️ Weather in {data['city']}:\n"
                f"Temperature: {data['temperature']}°C\n"
                f"Humidity: {data['humidity']}%\n"
                f"Condition: {data['condition']}"
            )
        else:
            return "❌ Could not fetch weather."

    return "I can help with weather info!"

# ------------------------------
# DEMO RUN
# ------------------------------
print(weather_agent("What is the weather in Delhi"))
print(weather_agent("Tell me weather in Mumbai"))

❌ Could not fetch weather.
❌ Could not fetch weather.
